# 01 · Data cleaning
**Goal:** turn the raw BTS on-time file into an analysis-ready table with outcome definitions
that match what a business traveler experiences. All rules live in `src/prep.py` so the 200k
sample and the full 6.4M-flight files are cleaned identically.

In [1]:
import sys, json
sys.path.insert(0, "../src")
import numpy as np, pandas as pd
import prep
pd.set_option("display.width", 200, "display.max_columns", 60)

raw = prep.load_raw(prep.SAMPLE_CSV)
print(raw.shape)
raw.head(3)

(200000, 37)


,YEAR,QUARTER,MONTH,DAY_OF_MONTH,DAY_OF_WEEK,FL_DATE,OP_UNIQUE_CARRIER,OP_CARRIER,OP_CARRIER_FL_NUM,TAIL_NUM,ORIGIN,ORIGIN_CITY_NAME,ORIGIN_STATE_ABR,DEST,DEST_CITY_NAME,DEST_STATE_ABR,CRS_DEP_TIME,DEP_TIME,DEP_DELAY,DEP_DELAY_NEW,CRS_ARR_TIME,ARR_TIME,ARR_DELAY,ARR_DELAY_NEW,CANCELLED,CANCELLATION_CODE,DIVERTED,CRS_ELAPSED_TIME,ACTUAL_ELAPSED_TIME,AIR_TIME,DISTANCE,CARRIER_DELAY,WEATHER_DELAY,NAS_DELAY,SECURITY_DELAY,LATE_AIRCRAFT_DELAY,sample_weight
0,2025,1,2,25,2,2025-02-25,UA,UA,648,N14118,EWR,"Newark, NJ",NJ,SFO,"San Francisco, CA",CA,1400,1401,1.0,1.0,1740,1720,-20.0,0.0,0.0,NaN,0.0,400.0,379.0,355.0,2565.0,NaN,NaN,NaN,NaN,NaN,31.99028
1,2025,3,8,20,3,2025-08-20,AA,AA,2257,N845NN,ORD,"Chicago, IL",IL,DFW,"Dallas/Fort Worth, TX",TX,1455,1534,39.0,39.0,1725,1758,33.0,33.0,0.0,NaN,0.0,150.0,144.0,112.0,802.0,2.0,0.0,0.0,0.0,31.0,31.99028
2,2025,4,11,24,1,2025-11-24,DL,DL,1052,N321DH,MSP,"Minneapolis, MN",MN,ATL,"Atlanta, GA",GA,0855,0850,-5.0,0.0,1227,1214,-13.0,0.0,0.0,NaN,0.0,152.0,144.0,118.0,907.0,NaN,NaN,NaN,NaN,NaN,31.99028


## 1. Audit the raw file

In [2]:
nulls = raw.isna().sum()
audit = pd.DataFrame({"dtype": raw.dtypes.astype(str), "nulls": nulls,
                      "null_pct": (nulls / len(raw) * 100).round(2), "unique": raw.nunique()})
audit

,dtype,nulls,null_pct,unique
YEAR,int64,0,0.00,1
QUARTER,int64,0,0.00,4
MONTH,int64,0,0.00,12
DAY_OF_MONTH,int64,0,0.00,31
DAY_OF_WEEK,int64,0,0.00,7
FL_DATE,object,0,0.00,365
OP_UNIQUE_CARRIER,object,0,0.00,10
OP_CARRIER,object,0,0.00,10
OP_CARRIER_FL_NUM,string,0,0.00,6398
TAIL_NUM,object,248,0.12,5412


In [3]:
key = ["FL_DATE", "OP_UNIQUE_CARRIER", "OP_CARRIER_FL_NUM", "ORIGIN", "DEST", "CRS_DEP_TIME"]
comp = raw[(raw.CANCELLED == 0) & (raw.DIVERTED == 0)]
checks = {
    "full-row duplicates": int(raw.duplicated().sum()),
    "duplicate flight keys": int(raw.duplicated(key).sum()),
    "OP_CARRIER differs from OP_UNIQUE_CARRIER": int((raw.OP_UNIQUE_CARRIER != raw.OP_CARRIER).sum()),
    "ARR_DELAY missing on completed flights": int(comp.ARR_DELAY.isna().sum()),
    "ARR_DELAY present on cancelled/diverted flights": int(raw[(raw.CANCELLED == 1) | (raw.DIVERTED == 1)].ARR_DELAY.notna().sum()),
    "cause columns present when delay < 15 min": int(comp[comp.ARR_DELAY < 15].CARRIER_DELAY.notna().sum()),
    "cause columns missing when delay >= 15 min": int(comp[comp.ARR_DELAY >= 15].CARRIER_DELAY.isna().sum()),
    "'2400' in DEP_TIME / ARR_TIME": int((raw.DEP_TIME == "2400").sum() + (raw.ARR_TIME == "2400").sum()),
    "non-positive DISTANCE or AIR_TIME": int((raw.DISTANCE <= 0).sum() + (raw.AIR_TIME <= 0).sum()),
    "arrival delay > 1,000 min": int((raw.ARR_DELAY > 1000).sum()),
    "distinct sample_weight values": int(raw.sample_weight.nunique()),
}
pd.Series(checks, name="rows").to_frame()

,rows
full-row duplicates,0
duplicate flight keys,0
OP_CARRIER differs from OP_UNIQUE_CARRIER,0
ARR_DELAY missing on completed flights,0
ARR_DELAY present on cancelled/diverted flights,0
cause columns present when delay < 15 min,0
cause columns missing when delay >= 15 min,0
'2400' in DEP_TIME / ARR_TIME,152
non-positive DISTANCE or AIR_TIME,0
"arrival delay > 1,000 min",91


**Reading the audit.** Every missing value is *structural*, not an error:
* `ARR_DELAY` is blank only when a flight was cancelled or diverted - the worst outcomes for a
  traveler. Dropping those rows would make every carrier look better than it is, so we keep them
  and count them as disruptions.
* The five delay-cause columns are only filled when a flight arrives 15+ minutes late.
* `sample_weight` is one constant (31.99), so unweighted rates equal population rates.

## 2. Apply the cleaning rules

In [4]:
df = prep.clean(raw)
print(df.shape)
df[["FL_DATE", "CARRIER", "route", "sched_dep", "sched_arr", "act_arr", "ARR_DELAY", "disrupted",
    "severe", "time_band", "late_night_arrival", "regional", "holiday_window"]].head(5)

(200000, 57)


,FL_DATE,CARRIER,route,sched_dep,sched_arr,act_arr,ARR_DELAY,disrupted,severe,time_band,late_night_arrival,regional,holiday_window
0,2025-02-25,UA,EWR-SFO,2025-02-25 14:00:00,2025-02-25 17:40:00,2025-02-25 17:20:00,-20.0,0,0,Afternoon (2-5:59p),0,0,0
1,2025-08-20,AA,ORD-DFW,2025-08-20 14:55:00,2025-08-20 17:25:00,2025-08-20 17:58:00,33.0,1,0,Afternoon (2-5:59p),0,0,0
2,2025-11-24,DL,MSP-ATL,2025-11-24 08:55:00,2025-11-24 12:27:00,2025-11-24 12:14:00,-13.0,0,0,Morning (7-9:59a),0,0,1
3,2025-10-07,MQ,DFW-JAN,2025-10-07 07:04:00,2025-10-07 08:39:00,2025-10-07 08:15:00,-24.0,0,0,Morning (7-9:59a),0,1,0
4,2025-08-27,OH,DFW-LAW,2025-08-27 17:13:00,2025-08-27 18:11:00,2025-08-27 18:45:00,34.0,1,0,Afternoon (2-5:59p),0,1,0


### Outcome definitions (used in every later notebook)
| Measure | Definition | Why |
|---|---|---|
| **Disrupted** | arrived 15+ min late, **or** cancelled, **or** diverted | DOT's on-time standard, plus the outcomes that have no arrival delay |
| **Severely disrupted** | arrived 60+ min late, or cancelled, or diverted | "missed the meeting" |
| Arrival delay (min) | completed flights only | used for buffer-time rules (p90 / p95) |
| Traveler burden | departure before 6 a.m.; landing 11 p.m.-4 a.m.; 2h+ departure delay; weekend travel | the employee's side of the same events |

In [5]:
# overnight / time-zone handling check: scheduled arrival minus departure should equal
# scheduled elapsed time plus a whole-hour time-zone shift
shift = ((df.sched_arr - df.sched_dep).dt.total_seconds() / 60 - df.CRS_ELAPSED_TIME)
print("implied time-zone shifts (hours) and share of flights:")
print((shift / 60).round(2).value_counts(normalize=True).head(12).round(4))
print("\nflights whose scheduled arrival is on the next calendar day:",
      int((df.sched_arr.dt.normalize() > df.FL_DATE).sum()))

implied time-zone shifts (hours) and share of flights:
 0.0    0.4955
 1.0    0.1631
-1.0    0.1629
-2.0    0.0531
 2.0    0.0527
-3.0    0.0361
 3.0    0.0346
 4.0    0.0006
-4.0    0.0005
-5.0    0.0003
 5.0    0.0002
 6.0    0.0002
Name: proportion, dtype: float64

flights whose scheduled arrival is on the next calendar day: 6203


## 3. Validate against the organisers' report

In [6]:
rep = json.load(open(prep.VALIDATION_JSON))
val = pd.DataFrame([
    ("rows", rep["student_records"], len(df)),
    ("cancelled", rep["student_cancelled"], int(df.CANCELLED.sum())),
    ("diverted", rep["student_diverted"], int(df.DIVERTED.sum())),
    ("airports", rep["student_airports"], len(set(df.ORIGIN) | set(df.DEST))),
    ("carriers match", True, df.CARRIER.value_counts().to_dict() == rep["student_carriers"]),
    ("months match", True, {str(k): v for k, v in df.MONTH.value_counts().items()} == rep["student_months"]),
], columns=["check", "expected", "ours"])
val["ok"] = val.expected == val.ours
val

,check,expected,ours,ok
0,rows,200000,200000,True
1,cancelled,2907,2907,True
2,diverted,570,570,True
3,airports,327,327,True
4,carriers match,True,True,True
5,months match,True,True,True


## 4. Cleaning log

In [7]:
log = pd.DataFrame([
    ("Times stored as HHMM numbers", "Read as text, convert to minutes after midnight; '2400' -> '0000'", len(df)),
    ("Arrival date not given", "Rebuilt scheduled/actual arrival timestamps from schedule (handles overnight + time zones)", len(df)),
    ("ARR_DELAY blank for cancelled flights", "Kept; counted as disrupted and severely disrupted", int(df.CANCELLED.sum())),
    ("ARR_DELAY blank for diverted flights", "Kept; counted as disrupted and severely disrupted", int(df.DIVERTED.sum())),
    ("Delay-cause columns blank when delay < 15 min", "Left blank; cause analysis uses delayed flights only", int(df.CARRIER_DELAY.isna().sum())),
    ("OP_CARRIER duplicates OP_UNIQUE_CARRIER", "Dropped duplicate; renamed to CARRIER", len(df)),
    ("Extreme delays (> 1,000 min)", "Kept (real events); report medians/percentiles beside means", int((df.ARR_DELAY > 1000).sum())),
    ("TAIL_NUM blank", "Kept; only occurs on cancelled flights; not used as a feature", int(df.TAIL_NUM.isna().sum())),
    ("Rows removed", "None", 0),
], columns=["issue", "rule", "rows_affected"])
log.to_csv(prep.TABLES / "cleaning_log.csv", index=False)
log

,issue,rule,rows_affected
0,Times stored as HHMM numbers,"Read as text, convert to minutes after midnigh...",200000
1,Arrival date not given,Rebuilt scheduled/actual arrival timestamps fr...,200000
2,ARR_DELAY blank for cancelled flights,Kept; counted as disrupted and severely disrupted,2907
3,ARR_DELAY blank for diverted flights,Kept; counted as disrupted and severely disrupted,570
4,Delay-cause columns blank when delay < 15 min,Left blank; cause analysis uses delayed flight...,156488
5,OP_CARRIER duplicates OP_UNIQUE_CARRIER,Dropped duplicate; renamed to CARRIER,200000
6,"Extreme delays (> 1,000 min)",Kept (real events); report medians/percentiles...,91
7,TAIL_NUM blank,Kept; only occurs on cancelled flights; not us...,248
8,Rows removed,None,0


In [8]:
prep.PROCESSED.mkdir(parents=True, exist_ok=True)
df.to_parquet(prep.PROCESSED / "flights_sample.parquet", index=False)
base = df[["disrupted", "severe", "CANCELLED", "DIVERTED"]].mean().rename("rate").to_frame()
base.loc["on_time_of_completed"] = (df.loc[df.completed == 1, "ARR_DELAY"] < 15).mean()
base.round(4)

,rate
disrupted,0.2349
severe,0.0974
CANCELLED,0.0145
DIVERTED,0.0028
on_time_of_completed,0.7786
